In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
import os
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive")

# Change this after checking your Drive structure
DAIC_DIR = BASE_DIR / "EDAIC"

print("Exists:", DAIC_DIR.exists())
print("Path:", DAIC_DIR)

Exists: True
Path: /content/drive/MyDrive/EDAIC


In [10]:
for item in DAIC_DIR.iterdir():
    print(item)

/content/drive/MyDrive/EDAIC/302_Transcript.gsheet
/content/drive/MyDrive/EDAIC/363_BoAW_openSMILE_2.3.0_eGeMAPS.gsheet
/content/drive/MyDrive/EDAIC/364_OpenFace2.1.0_Pose_gaze_AUs.gsheet
/content/drive/MyDrive/EDAIC/411_OpenFace2.1.0_Pose_gaze_AUs.csv
/content/drive/MyDrive/EDAIC/411_OpenSMILE2.3.0_egemaps.csv
/content/drive/MyDrive/EDAIC/411_BoVW_openFace_2.1.0_Pose_Gaze_AUs.csv
/content/drive/MyDrive/EDAIC/411_BoAW_openSMILE_2.3.0_eGeMAPS.csv
/content/drive/MyDrive/EDAIC/413_Transcript.csv
/content/drive/MyDrive/EDAIC/413_OpenFace2.1.0_Pose_gaze_AUs.csv
/content/drive/MyDrive/EDAIC/413_OpenSMILE2.3.0_egemaps.csv
/content/drive/MyDrive/EDAIC/413_BoVW_openFace_2.1.0_Pose_Gaze_AUs.csv
/content/drive/MyDrive/EDAIC/413_BoAW_openSMILE_2.3.0_eGeMAPS.csv
/content/drive/MyDrive/EDAIC/414_Transcript.csv
/content/drive/MyDrive/EDAIC/414_OpenFace2.1.0_Pose_gaze_AUs.csv
/content/drive/MyDrive/EDAIC/414_OpenSMILE2.3.0_egemaps.csv
/content/drive/MyDrive/EDAIC/414_BoVW_openFace_2.1.0_Pose_Gaze_AUs.

In [12]:
openface_files = []

for root, dirs, files in os.walk(BASE_DIR):
    for file in files:
        if "OpenFace" in file and "Pose" in file and "AUs" in file and file.endswith(".csv"):
            openface_files.append(os.path.join(root, file))

print("OpenFace files found:", len(openface_files))

for f in openface_files[:20]:
    print(f)

OpenFace files found: 163
/content/drive/MyDrive/EDAIC/411_OpenFace2.1.0_Pose_gaze_AUs.csv
/content/drive/MyDrive/EDAIC/413_OpenFace2.1.0_Pose_gaze_AUs.csv
/content/drive/MyDrive/EDAIC/414_OpenFace2.1.0_Pose_gaze_AUs.csv
/content/drive/MyDrive/EDAIC/416_OpenFace2.1.0_Pose_gaze_AUs.csv
/content/drive/MyDrive/EDAIC/417_OpenFace2.1.0_Pose_gaze_AUs.csv
/content/drive/MyDrive/EDAIC/418_OpenFace2.1.0_Pose_gaze_AUs.csv
/content/drive/MyDrive/EDAIC/419_OpenFace2.1.0_Pose_gaze_AUs.csv
/content/drive/MyDrive/EDAIC/420_OpenFace2.1.0_Pose_gaze_AUs.csv
/content/drive/MyDrive/EDAIC/421_OpenFace2.1.0_Pose_gaze_AUs.csv
/content/drive/MyDrive/EDAIC/422_OpenFace2.1.0_Pose_gaze_AUs.csv
/content/drive/MyDrive/EDAIC/424_OpenFace2.1.0_Pose_gaze_AUs.csv
/content/drive/MyDrive/EDAIC/426_OpenFace2.1.0_Pose_gaze_AUs.csv
/content/drive/MyDrive/EDAIC/427_OpenFace2.1.0_Pose_gaze_AUs.csv
/content/drive/MyDrive/EDAIC/428_OpenFace2.1.0_Pose_gaze_AUs.csv
/content/drive/MyDrive/EDAIC/429_OpenFace2.1.0_Pose_gaze_AUs.csv

In [13]:
import pandas as pd
import re

records = []

for path in openface_files:
    filename = os.path.basename(path)

    match = re.search(r"(\d+)_OpenFace", filename)
    participant_id = int(match.group(1)) if match else None

    records.append({
        "Participant_ID": participant_id,
        "filename": filename,
        "path": path
    })

video_files_df = pd.DataFrame(records)
video_files_df = video_files_df.sort_values("Participant_ID").reset_index(drop=True)

video_files_df.head()

,Participant_ID,filename,path
0,302,302_OpenFace2.1.0_Pose_gaze_AUs.csv,/content/drive/MyDrive/edaic/302/features/302_...
1,303,303_OpenFace2.1.0_Pose_gaze_AUs.csv,/content/drive/MyDrive/edaic/303/features/303_...
2,304,304_OpenFace2.1.0_Pose_gaze_AUs.csv,/content/drive/MyDrive/edaic/304/features/304_...
3,305,305_OpenFace2.1.0_Pose_gaze_AUs.csv,/content/drive/MyDrive/edaic/305/features/305_...
4,307,307_OpenFace2.1.0_Pose_gaze_AUs.csv,/content/drive/MyDrive/edaic/307/features/307_...


In [14]:
BASE_DIR

PosixPath('/content/drive/MyDrive')

In [15]:
OUT_DIR = BASE_DIR / "DAIC_WOZ_VIDEO_BRANCH" / "results" / "tables"
OUT_DIR.mkdir(parents=True, exist_ok=True)

video_files_df.to_csv(OUT_DIR / "video_file_availability.csv", index=False)

In [18]:
import os
import requests

LABELS_URL = "https://dcapswoz.ict.usc.edu/wwwedaic/labels/"
SAVE_DIR = "/content/drive/MyDrive/EDAIC/labels"

os.makedirs(SAVE_DIR, exist_ok=True)

LABEL_FILES = [
    "train_split.csv",
    "dev_split.csv",
    "test_split.csv",
    "Detailed_PHQ8_Labels.csv"
]

def download_label_files():
    for fname in LABEL_FILES:
        url = LABELS_URL + fname
        out_path = os.path.join(SAVE_DIR, fname)

        if os.path.exists(out_path):
            print(f"✓ Already exists: {fname}")
            continue

        r = requests.get(url)

        if r.status_code == 200:
            with open(out_path, "wb") as f:
                f.write(r.content)
            print(f"✓ Downloaded: {fname}")
        else:
            print(f"✗ Failed: {fname} | Status code: {r.status_code}")

# Run
download_label_files()

✓ Downloaded: train_split.csv
✓ Downloaded: dev_split.csv
✓ Downloaded: test_split.csv
✓ Downloaded: Detailed_PHQ8_Labels.csv


In [19]:
label_files = []

for root, dirs, files in os.walk(DAIC_DIR):
    for file in files:
        if (
            "split" in file.lower()
            or "phq" in file.lower()
            or "depression" in file.lower()
        ) and file.endswith(".csv"):
            label_files.append(os.path.join(root, file))

print("Label files found:", len(label_files))

for f in label_files:
    print(f)

Label files found: 4
/content/drive/MyDrive/EDAIC/labels/train_split.csv
/content/drive/MyDrive/EDAIC/labels/dev_split.csv
/content/drive/MyDrive/EDAIC/labels/test_split.csv
/content/drive/MyDrive/EDAIC/labels/Detailed_PHQ8_Labels.csv


In [20]:
for f in label_files:
    print("\nFILE:", f)
    df = pd.read_csv(f)
    print(df.shape)
    print(df.columns.tolist())
    display(df.head())


FILE: /content/drive/MyDrive/EDAIC/labels/train_split.csv
(163, 6)
['Participant_ID', 'Gender', 'PHQ_Binary', 'PHQ_Score', 'PCL-C (PTSD)', 'PTSD Severity']


,Participant_ID,Gender,PHQ_Binary,PHQ_Score,PCL-C (PTSD),PTSD Severity
0,302,male,0,4,0,28
1,303,female,0,0,0,17
2,304,female,0,6,0,20
3,305,male,0,7,0,28
4,307,female,0,4,0,23



FILE: /content/drive/MyDrive/EDAIC/labels/dev_split.csv
(56, 6)
['Participant_ID', 'Gender', 'PHQ_Binary', 'PHQ_Score', 'PCL-C (PTSD)', 'PTSD Severity']


,Participant_ID,Gender,PHQ_Binary,PHQ_Score,PCL-C (PTSD),PTSD Severity
0,300,male,0,2,0,25
1,301,male,0,3,0,17
2,306,female,0,0,0,21
3,317,male,0,8,1,51
4,320,female,0,11,1,64



FILE: /content/drive/MyDrive/EDAIC/labels/test_split.csv
(56, 6)
['Participant_ID', 'Gender', 'PHQ_Binary', 'PHQ_Score', 'PCL-C (PTSD)', 'PTSD Severity']


,Participant_ID,Gender,PHQ_Binary,PHQ_Score,PCL-C (PTSD),PTSD Severity
0,600,female,0,5,0,23.0
1,602,female,1,13,1,67.0
2,604,male,1,12,0,30.0
3,605,male,0,2,0,23.0
4,606,female,0,5,0,46.0



FILE: /content/drive/MyDrive/EDAIC/labels/Detailed_PHQ8_Labels.csv
(219, 10)
['Participant_ID', 'PHQ_8NoInterest', 'PHQ_8Depressed', 'PHQ_8Sleep', 'PHQ_8Tired', 'PHQ_8Appetite', 'PHQ_8Failure', 'PHQ_8Concentrating', 'PHQ_8Moving', 'PHQ_8Total']


,Participant_ID,PHQ_8NoInterest,PHQ_8Depressed,PHQ_8Sleep,PHQ_8Tired,PHQ_8Appetite,PHQ_8Failure,PHQ_8Concentrating,PHQ_8Moving,PHQ_8Total
0,300,0,0,1,0,1,0,0,0,2
1,301,0,0,1,1,1,0,0,0,3
2,302,1,1,0,1,0,1,0,0,4
3,303,0,0,0,0,0,0,0,0,0
4,304,0,1,1,2,2,0,0,0,6


In [22]:
train_df = pd.read_csv(f"{DAIC_DIR}/labels/train_split.csv")
dev_df = pd.read_csv(f"{DAIC_DIR}/labels/dev_split.csv")
# dev_df = pd.read_csv("PATH_TO_dev_split_Depression_AVEC2017.csv")

train_df["split"] = "train"
dev_df["split"] = "dev"

labels_df = pd.concat([train_df, dev_df], ignore_index=True)

labels_df.head()

,Participant_ID,Gender,PHQ_Binary,PHQ_Score,PCL-C (PTSD),PTSD Severity,split
0,302,male,0,4,0,28,train
1,303,female,0,0,0,17,train
2,304,female,0,6,0,20,train
3,305,male,0,7,0,28,train
4,307,female,0,4,0,23,train


In [23]:
labels_df.columns = [c.strip() for c in labels_df.columns]

print(labels_df.columns)

Index(['Participant_ID', 'Gender', 'PHQ_Binary', 'PHQ_Score', 'PCL-C (PTSD)',
       'PTSD Severity', 'split'],
      dtype='object')


In [25]:
labels_df[["Participant_ID", "split", "PHQ_Binary", "PHQ_Score"]].head()

,Participant_ID,split,PHQ_Binary,PHQ_Score
0,302,train,0,4
1,303,train,0,0
2,304,train,0,6
3,305,train,0,7
4,307,train,0,4


In [26]:
LABEL_OUT = BASE_DIR / "DAIC_WOZ_VIDEO_BRANCH" / "data" / "labels"
LABEL_OUT.mkdir(parents=True, exist_ok=True)

labels_clean = labels_df[["Participant_ID", "split", "PHQ_Binary", "PHQ_Score"]].copy()
labels_clean.to_csv(LABEL_OUT / "labels_train_dev_clean.csv", index=False)

In [27]:
sample_path = video_files_df.iloc[0]["path"]
sample_df = pd.read_csv(sample_path)

print(sample_df.shape)
print(sample_df.columns.tolist())
sample_df.head()

(22766, 53)
['frame', 'timestamp', 'confidence', 'success', 'pose_Tx', 'pose_Ty', 'pose_Tz', 'pose_Rx', 'pose_Ry', 'pose_Rz', 'gaze_0_x', 'gaze_0_y', 'gaze_0_z', 'gaze_1_x', 'gaze_1_y', 'gaze_1_z', 'gaze_angle_x', 'gaze_angle_y', 'AU01_r', 'AU02_r', 'AU04_r', 'AU05_r', 'AU06_r', 'AU07_r', 'AU09_r', 'AU10_r', 'AU12_r', 'AU14_r', 'AU15_r', 'AU17_r', 'AU20_r', 'AU23_r', 'AU25_r', 'AU26_r', 'AU45_r', 'AU01_c', 'AU02_c', 'AU04_c', 'AU05_c', 'AU06_c', 'AU07_c', 'AU09_c', 'AU10_c', 'AU12_c', 'AU14_c', 'AU15_c', 'AU17_c', 'AU20_c', 'AU23_c', 'AU25_c', 'AU26_c', 'AU28_c', 'AU45_c']


,frame,timestamp,confidence,success,pose_Tx,pose_Ty,pose_Tz,pose_Rx,pose_Ry,pose_Rz,...,AU12_c,AU14_c,AU15_c,AU17_c,AU20_c,AU23_c,AU25_c,AU26_c,AU28_c,AU45_c
0,1,0.000,0.98,1,4.2,19.5,565.7,0.238,0.154,-0.061,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,2,0.033,0.98,1,4.2,19.2,562.7,0.225,0.145,-0.061,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,0.067,0.98,1,4.2,19.1,562.5,0.226,0.144,-0.060,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,0.100,0.98,1,4.2,19.1,562.4,0.227,0.144,-0.060,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,0.133,0.98,1,4.2,19.1,562.3,0.227,0.144,-0.060,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
